In [1]:
import msgpack, msgpack_numpy, numpy as np, requests
from droid_plus.policies.image_tools import resize_with_pad

POLICY_URL = "http://127.0.0.1:8000"     # your SSH tunnel endpoint
PROMPT     = "lift the white color cuboid object from the table"

print("health:", requests.get(f"{POLICY_URL}/health", timeout=10).json())

def infer(left, wrist, q, grip, prompt=PROMPT):
    req = {
        "images": {
            "left":  resize_with_pad(np.asarray(left,  np.uint8), 224, 224),
            "wrist": resize_with_pad(np.asarray(wrist, np.uint8), 224, 224),
        },
        "state":  np.concatenate([np.asarray(q, np.float32).reshape(-1), np.float32([grip])]),
        "prompt": prompt,
    }
    blob = msgpack.packb(req, default=msgpack_numpy.encode, use_bin_type=True)
    r = requests.post(f"{POLICY_URL}/infer", data=blob,
                      headers={"Content-Type": "application/msgpack"}, timeout=20)
    r.raise_for_status()
    out = msgpack.unpackb(r.content, object_hook=msgpack_numpy.decode, raw=False)
    return np.asarray(out["actions"], dtype=float)     # (10, 8)


health: {'ok': True, 'checkpoint': '../outputs/pi05_fr3_lift/checkpoints/last/pretrained_model/', 'device': 'cuda', 'action_dim': 8}


In [2]:
from droid_plus.robot import DroidPlus
droid = DroidPlus()
g = droid.gripper

left  = droid.get_left_image(jpeg_quality=90)         # RGB uint8
wrist = droid.get_wrist_image(jpeg_quality=90)
q     = np.asarray(droid.get_current_joint_state()["positions"], np.float32)
try:
    grip = float(g.gripper_position_frac())           # 0=open, 1=closed
except Exception:
    grip = 0.0

print("current q :", np.round(q, 3), " grip:", round(grip, 3))

chunk = infer(left, wrist, q, grip)
# print("chunk shape:", chunk.shape)
# print("action[0]  :", np.round(chunk[0], 3))
# print("|action[0][:7] - q| :", np.round(np.abs(chunk[0,:7] - q), 3), " max:", round(np.abs(chunk[0,:7]-q).max(), 3))
# print("gripper col:", np.round(chunk[:,7], 2))
# print("per-step Δ (rad):", np.round(np.abs(np.diff(chunk[:,:7], axis=0)).max(axis=1), 3))


gripper initialized None
current q : [-0.154  0.456  0.095 -2.412 -0.143  2.694  0.884]  grip: 0.0


In [3]:
import time

CONTROL_HZ    = 15.0
EXECUTE_STEPS = 16          # run ~1 s of the chunk, then re-infer
MAX_CHUNKS    = 40
MAX_STEP      = 0.05        # rad/tick clamp toward each target (absorbs the q->action[0] gap)
STEP_ABORT    = 0.25        # abort a chunk whose own steps exceed this
DT = 1.0 / CONTROL_HZ

Q_MIN = np.array([-2.7437,-1.7837,-2.9007,-3.0421,-2.8065, 0.5445,-3.0159])
Q_MAX = np.array([ 2.7437, 1.7837, 2.9007,-0.1518, 2.8065, 4.5169, 3.0159])
MARGIN = 0.12

GRIP_SPEED, GRIP_FORCE = 120, 80
APPROVE_FIRST_CLOSE = True
DRY_RUN = False             # print only; no arm/gripper commands

def get_obs():
    left  = droid.get_left_image(jpeg_quality=90)
    wrist = droid.get_wrist_image(jpeg_quality=90)
    q     = np.asarray(droid.get_current_joint_state()["positions"], float)
    try:    grip = float(g.gripper_position_frac())
    except Exception: grip = 0.0
    return left, wrist, q, grip

def send_q(qc, seq):
    if not DRY_RUN:
        droid.set_target_joint_state(np.clip(qc, Q_MIN, Q_MAX), velocities=[0.0]*7, seq=seq)

def soft_stop():
    time.sleep(0.15)                    # let franky's 100 ms watchdog decay first
    if not DRY_RUN:
        try: droid.stop()
        except Exception: pass


In [4]:
# ============================================================================
#  TABLE-COLLISION SAFETY  -  keep the gripper >= TABLE_CLEARANCE above the table
# ============================================================================
# Dependency-free analytic forward kinematics for the Franka FR3 (modified DH).
# We track the base-frame height (Z) of the gripper tip and refuse any commanded
# joint step that would drive it below  table_surface + TABLE_CLEARANCE.
# Verified: q=[0,-0.785,0,-2.356,0,1.571,0.785] -> panda_link8 = [0.307, 0, 0.590].

TABLE_CLEARANCE = 0.015     # 1.5 cm hard floor above the table surface
TCP_OFFSET_Z    = 0.16      # metres from the flange (panda_link8) to the gripper
                            # tip, along the flange local +Z. Robotiq 2F-85 on the
                            # Franka mount ~= 0.15-0.17 m. The calibration cell
                            # below makes the exact value non-critical (the same
                            # offset is used at calibration and at run time).

_FR3_DH = [                 # (a, d, alpha);  theta = q_i  (last row = fixed flange)
    (0.0,     0.333,  0.0),
    (0.0,     0.0,   -np.pi / 2),
    (0.0,     0.316,  np.pi / 2),
    (0.0825,  0.0,    np.pi / 2),
    (-0.0825, 0.384, -np.pi / 2),
    (0.0,     0.0,    np.pi / 2),
    (0.088,   0.0,    np.pi / 2),
    (0.0,     0.107,  0.0),
]

def _fr3_flange_T(q):
    """4x4 base->flange (panda_link8) transform for arm joints q (7,)."""
    q = np.asarray(q, float).reshape(-1)[:7]
    T = np.eye(4)
    for (a, d, alpha), th in zip(_FR3_DH, list(q) + [0.0]):
        ca, sa, ct, st = np.cos(alpha), np.sin(alpha), np.cos(th), np.sin(th)
        T = T @ np.array([
            [ct,    -st,    0.0,  a],
            [st*ca,  ct*ca, -sa, -sa*d],
            [st*sa,  ct*sa,  ca,  ca*d],
            [0.0,    0.0,    0.0,  1.0],
        ])
    return T

def gripper_z(q):
    """Base-frame height (m) of the gripper tip for arm joint vector q (7,)."""
    T = _fr3_flange_T(q)
    return float(T[2, 3] + T[2, 2] * TCP_OFFSET_Z)   # flange origin + Z_flange * offset

# --- table height in this FK frame ------------------------------------------
# Either set TABLE_Z directly (if you measured the tip height in base frame) or
# run the calibration cell below.
TABLE_Z = None

def z_floor():
    if TABLE_Z is None:
        raise RuntimeError("TABLE_Z is not set - run the table-height calibration cell")
    return TABLE_Z + TABLE_CLEARANCE


In [ ]:
# --- CALIBRATE the table height -------------------------------------------------
# Jog the arm (teleop) until the gripper tip just rests on the table surface,
# then run this cell ONCE. Re-run whenever the gripper/mount changes.
#
# Alternative without touching the table: hold the tip a known height H above the
# table, run the two lines, then do  TABLE_Z = TABLE_Z - H.

# _qcal = get_obs()[2]
# TABLE_Z = gripper_z(_qcal)
# print(f"calibrated TABLE_Z = {TABLE_Z:+.4f} m   from q = {np.round(_qcal, 3)}")
# print(f"safety floor       = {z_floor():+.4f} m   (= table + {TABLE_CLEARANCE*100:.1f} cm)")
# print(f"gripper tip now     = {gripper_z(get_obs()[2]):+.4f} m")


calibrated TABLE_Z = -0.0441 m   from q = [-0.154  0.456  0.095 -2.412 -0.143  2.694  0.884]
safety floor       = -0.0291 m   (= table + 1.5 cm)
gripper tip now     = -0.0441 m


In [12]:
if not DRY_RUN:
    g.open(speed=GRIP_SPEED)
    droid.robot.set_command_timeout(3.0)

q0 = get_obs()[2]
START_Q = np.array([0.,0.,0.,-1.571,0.,1.571,0.])   # your teleop start pose == franky HOME
print("current q:", np.round(q0,3), " -> START_Q gap:", round(np.abs(START_Q-q0).max(),3))
input("workspace clear, E-stop in hand — Enter to reset to start ")

sent = q0.astype(float); seq = 0
try:
    while np.abs(START_Q - sent).max() > 1e-3:
        sent = np.clip(sent + np.clip(START_Q - sent, -0.03, 0.03), Q_MIN, Q_MAX)
        send_q(sent, seq); seq += 1; time.sleep(DT)
    for _ in range(8): send_q(sent, seq); seq += 1; time.sleep(DT)
finally:
    soft_stop()
print("at start:", np.round(get_obs()[2], 3))


current q: [-0.433  0.06   0.028 -2.385 -0.165  2.309  0.227]  -> START_Q gap: 0.814
at start: [-0.    -0.    -0.    -1.572 -0.     1.572  0.   ]


In [11]:
_grip = {"closed": None}
seq = 1000
Z_FLOOR = z_floor()   # raises if the table-height calibration cell hasn't run
print(f"table-collision guard active: gripper tip must stay >= {Z_FLOOR:+.4f} m")
try:
    for ci in range(1, MAX_CHUNKS + 1):
        left, wrist, q, grip = get_obs()
        chunk = infer(left, wrist, q, grip)

        raw = chunk[:EXECUTE_STEPS, :7]
        tgt = np.clip(raw, Q_MIN + MARGIN, Q_MAX - MARGIN)
        gplan = chunk[:EXECUTE_STEPS, 7]

        step_max = np.abs(np.diff(np.vstack([q, tgt]), axis=0)).max()
        if step_max > STEP_ABORT:
            raise RuntimeError(f"chunk step {step_max:.3f} rad > STEP_ABORT")
        if not np.allclose(raw, tgt, atol=1e-6):
            print(f"   (clipped {int((raw!=tgt).sum())} target values to joint limits)")

        # --- table-collision guard: lowest gripper tip over the planned chunk ---
        z_plan = np.array([gripper_z(tt) for tt in tgt])
        z_now  = gripper_z(q)
        if z_plan.min() < Z_FLOOR:
            raise RuntimeError(
                f"TABLE SAFETY: chunk {ci} would lower the gripper tip to "
                f"z={z_plan.min():+.3f} m < floor {Z_FLOOR:+.3f} m - stopping")

        close = bool(gplan.max() > 0.5)
        print(f"[{ci:02d}/{MAX_CHUNKS}] first_gap={np.abs(tgt[0]-q).max():.3f} "
              f"reach={np.abs(tgt[-1]-q).max():.3f} grip={gplan.min():.2f}..{gplan.max():.2f} "
              f"close={close} z_now={z_now:+.3f} z_plan_min={z_plan.min():+.3f}"
              + ("  [DRY]" if DRY_RUN else ""))

        if not DRY_RUN:
            if close and _grip["closed"] is not True:
                if APPROVE_FIRST_CLOSE and _grip["closed"] is None:
                    input(f"[chunk {ci}] policy wants CLOSE - Enter to allow, interrupt to abort ")
                g.close_async(speed=GRIP_SPEED, force=GRIP_FORCE); _grip["closed"] = True
            elif not close and _grip["closed"] is not False:
                g.open_async(speed=GRIP_SPEED); _grip["closed"] = False

        sent = q.copy(); t = time.time()
        for k in range(EXECUTE_STEPS):
            cand = np.clip(sent + np.clip(tgt[k] - sent, -MAX_STEP, MAX_STEP), Q_MIN, Q_MAX)
            z_cand = gripper_z(cand)
            if z_cand < Z_FLOOR:
                raise RuntimeError(
                    f"TABLE SAFETY: chunk {ci} step {k} would put the gripper tip at "
                    f"z={z_cand:+.3f} m < floor {Z_FLOOR:+.3f} m - stopping")
            sent = cand
            send_q(sent, seq); seq += 1
            t += DT; time.sleep(max(0.0, t - time.time()))
    else:
        print("MAX_CHUNKS reached")
except KeyboardInterrupt:
    print("interrupted")
except Exception as e:
    print("ABORT:", type(e).__name__, e)
finally:
    soft_stop()
    print("final q:", np.round(get_obs()[2], 3))


table-collision guard active: gripper tip must stay >= -0.0291 m
[01/40] first_gap=0.161 reach=0.312 grip=-0.00..0.01 close=False z_now=+0.465 z_plan_min=+0.285
[02/40] first_gap=0.067 reach=0.423 grip=-0.00..0.00 close=False z_now=+0.300 z_plan_min=+0.157
[03/40] first_gap=0.026 reach=0.200 grip=-0.01..-0.00 close=False z_now=+0.176 z_plan_min=+0.069
[04/40] first_gap=0.035 reach=0.138 grip=0.00..0.00 close=False z_now=+0.078 z_plan_min=+0.034
[05/40] first_gap=0.015 reach=0.097 grip=0.00..0.01 close=False z_now=+0.041 z_plan_min=+0.004
[06/40] first_gap=0.024 reach=0.076 grip=0.00..0.01 close=False z_now=+0.010 z_plan_min=-0.018
[07/40] first_gap=0.016 reach=0.015 grip=0.01..0.50 close=False z_now=-0.015 z_plan_min=-0.012
[08/40] first_gap=0.020 reach=0.008 grip=0.01..0.53 close=True z_now=-0.012 z_plan_min=-0.010
[09/40] first_gap=0.011 reach=0.013 grip=0.01..0.53 close=True z_now=-0.009 z_plan_min=-0.007
[10/40] first_gap=0.008 reach=0.076 grip=0.57..0.58 close=True z_now=-0.007 z_